# Beer–Lambert spectra of A → B → C

*physchem · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/beer_lambert_kinetics)

A spectrophotometer follows the consecutive reaction $A \to B \to C$. The spectrum of a mixture is a matrix product, $D = l\,E\,c$ (wavelengths × species), so the concentration profiles come back from noisy spectra by least squares with the pseudo-inverse $E^{+}$. Watch the isosbestic points, where the spectra stay put while one step runs.

**Vector form**

$$D(\lambda, t) = l\sum_s \varepsilon_s(\lambda)\,c_s(t),\qquad \frac{d\mathbf c}{dt} = K\,\mathbf c,\qquad \hat{\mathbf c} = \frac{1}{l}\,(E^{\mathsf T}E)^{-1}E^{\mathsf T}\,\mathbf D$$

**Index (tensor) form**

$$D_{tw} = l\,E_{ws}\,c_{ts} \qquad c_{ts} = \big[e^{K t_t}\big]_{sr}\,c_r(0) \qquad G_{rs} = E_{wr}E_{ws},\;\; \hat c_{ts} = \tfrac{1}{l}\,(G^{-1})_{sr}\,E_{wr}\,D_{tw}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\lambda_w$ | `wavelengths(dev)` | (W,) | 300–700 nm in 2 nm steps |
| $E_{ws} = \varepsilon_s(\lambda_w)$ | `absorptivities(lam)` | (W, S) | molar absorptivity of each species (synthetic Gaussian bands, L mol$^{-1}$ cm$^{-1}$) |
| $K_{sr}$ | `rate_matrix(k1, k2)` | (S, S) | rate matrix of $A \to B \to C$; each column sums to 0 (mass balance) |
| $[e^{K t_t}]_{sr}\,c_r(0)$ | `einsum('tsr,r->ts', matrix_exp(K * t), c0)` | (T, S) | exact concentrations at every sampling time; $r$ summed |
| $l\,E_{ws}\,c_{ts}$ | `l * einsum('ws,ts->tw', E, c)` | (T, W) | Beer–Lambert for a mixture: absorbances add, species $s$ summed |
| $G_{rs} = E_{wr}E_{ws}$ | `einsum('wr,ws->rs', E, E)` | (S, S) | overlap (Gram) matrix of the pure spectra; $w$ summed |
| $E^{+}_{sw} = (G^{-1})_{sr}E_{wr}$ | `einsum('sr,wr->sw', inv(G), E)` | (S, W) | pseudo-inverse; equals torch.linalg.pinv(E) when the spectra are independent |
| $\hat c_{ts}$ | `einsum('sw,tw->ts', E_pinv, D) / l` | (T, S) | least-squares concentrations from each noisy spectrum; $w$ summed |
| $\varepsilon_A(\lambda) = \varepsilon_B(\lambda)$ | `crossings(lam, E[:, 0], E[:, 1])` | list | isosbestic point of the step $A \to B$ (likewise $B \to C$) |

Each species absorbs light in proportion to its concentration and the path length, $D = \varepsilon\,l\,c$, and the absorbances of a mixture simply add. Written for all wavelengths and times at once this is the matrix product $D_{tw} = l\,E_{ws}c_{ts}$, so with more wavelengths than species the concentrations follow from an overdetermined linear system, solved by least squares: $\hat c = E^{+}D/l$ with the pseudo-inverse $E^{+} = (E^{\mathsf T}E)^{-1}E^{\mathsf T}$. This only works when the pure spectra are clearly different; the condition number of $E$ printed in the terminal says how much the noise is amplified. The kinetics $A \to B \to C$ is a linear system $\dot{\mathbf c} = K\mathbf c$, solved exactly with the matrix exponential: $[A]$ decays, $[B]$ rises and falls (it peaks at $t = \ln(k_1/k_2)/(k_1 - k_2)$) and $[C]$ grows, while $[A] + [B] + [C]$ stays constant because every column of $K$ sums to zero. Isosbestic points: if only $A \to B$ is happening, $D = l(\varepsilon_A[A] + \varepsilon_B[B])$ with $[A] + [B]$ fixed, so at a wavelength where $\varepsilon_A = \varepsilon_B$ the absorbance cannot change. A sharp isosbestic point is the classic sign of a clean two-species conversion; here the crossing moves from $\varepsilon_A = \varepsilon_B$ (early) to $\varepsilon_B = \varepsilon_C$ (late), and in between it blurs because three species coexist. Set $k_2 = 0$ to see a perfect isosbestic point, or $k_2 \approx k_1$ to see it vanish. The spectra are synthetic Gaussian bands with typical dye-like absorptivities ($\sim 10^4$ L mol$^{-1}$ cm$^{-1}$), in a 1 cm cuvette.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/beer_lambert_kinetics/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k1, k2, length = params["k1"], params["k2"], tensors.PATH_LENGTH  # length = path length l (cm)
    lam = tensors.wavelengths(dev)  # (W,) nm
    E = tensors.absorptivities(lam)  # (W, S) L/(mol cm)
    c0 = tensors.initial_concentrations(params["c0"], dev)  # (S,) mol/L
    t = tensors.sample_times(params["t_end"], dev)  # (T,) s
    noise = tensors.noise((len(t), len(lam)), params["noise"], dev)  # (T, W)

    # 1) the experiment: kinetics → spectra (+ detector noise)
    c, D = solve.simulate(E, c0, t, k1, k2, length, noise)  # (T, S), (T, W)
    # 2) the analysis: spectra → concentrations by least squares
    c_hat, residual, cond = solve.unmix(E, D, length)

    uM = 1e6
    print(f"A → B → C   k1 = {k1:g} s⁻¹, k2 = {k2:g} s⁻¹   [A]0 = {params['c0']:g} µM, l = {length:g} cm")
    print(f"spectra: {len(t)} times × {len(lam)} wavelengths, noise σ = {params['noise']:g}   cond(E) = {cond:.2f}")
    t_peak = math.log(k1 / k2) / (k1 - k2) if (k2 > 0 and k1 != k2) else params["t_end"]
    print(f"B peaks at t ≈ {t_peak:.0f} s with {c[:, 1].max().item() * uM:.2f} µM")
    err = (c_hat - c).abs().max(0).values * uM
    print("max |ĉ − c| per species (µM): " + ", ".join(f"{s} {e:.3f}" for s, e in zip(tensors.SPECIES, err.tolist())))
    print(f"rms residual absorbance = {residual.pow(2).mean().sqrt().item():.4f}  (noise σ = {params['noise']:g})")
    total, total_hat = c.sum(1) * uM, c_hat.sum(1) * uM
    print(f"mass balance A+B+C: true {total.min().item():.6f}–{total.max().item():.6f} µM, "
          f"recovered {total_hat.min().item():.3f}–{total_hat.max().item():.3f} µM")  # fmt: skip

    # isosbestic points: where two spectra cross, the absorbance stays put while that step runs
    iso_ab = equation.crossings(lam, E[:, 0], E[:, 1])
    iso_bc = equation.crossings(lam, E[:, 1], E[:, 2])
    clean = equation.absorbance(E, c, length)  # (T, W) noise-free spectra
    early = t <= t_peak  # mostly A → B
    late = t >= 5.0 / k1  # A is gone (< 1 %): only B → C
    print("isosbestic points, change of absorbance ΔD in each phase (noise-free):")
    for name, points, phase in (("ε_A = ε_B", iso_ab, early), ("ε_B = ε_C", iso_bc, late)):
        if not phase.any():
            continue
        window = clean[phase]  # (T', W)
        swing = window.max(0).values - window.min(0).values  # (W,)
        span = f"t ≤ {t_peak:.0f} s" if phase is early else f"t ≥ {5.0 / k1:.0f} s"
        for x in points:
            w = int((lam - x).abs().argmin())
            print(f"  {name} at λ ≈ {x:.1f} nm, {span}:  ΔD = {swing[w].item():.4f}", end="")
            print(f"   (largest ΔD at any λ: {swing.max().item():.4f})")

    return plot.animate(lam, E, t, c, c_hat, D, iso_ab, iso_bc, out_dir / "beer_lambert_kinetics.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/beer_lambert_kinetics/equation.py
r"""Physics → tensor translation.

Beer–Lambert law for a mixture: every species s adds its own absorbance, so at wavelength w and time t

    D_{tw} = l E_{ws} c_{ts}          (s summed)       E_{ws} = ε_s(λ_w),  l = path length

The consecutive reaction A → B → C (rate constants k1, k2) is linear in the concentrations:

    dc_s/dt = K_{sr} c_r              (r summed)       K = [[-k1, 0, 0], [k1, -k2, 0], [0, k2, 0]]

so c_s(t) = [exp(K t)]_{sr} c_r(0). Every column of K sums to zero, which is mass balance: A + B + C = const.

Going back from spectra to concentrations is least squares, solved with the pseudo-inverse E⁺ = (Eᵀ E)⁻¹ Eᵀ:

    G_{rs} = E_{wr} E_{ws}  (w summed),     E⁺_{sw} = (G⁻¹)_{sr} E_{wr}  (r summed)
    ĉ_{ts} = E⁺_{sw} D_{tw} / l  (w summed)
"""

import torch


def absorbance(E: torch.Tensor, c: torch.Tensor, path_length: float) -> torch.Tensor:
    """D_{tw} = l E_{ws} c_{ts}.   E: (W, S), c: (T, S) → (T, W).  The species index s is summed."""
    return path_length * torch.einsum("ws,ts->tw", E, c)


def rate_matrix(k1: float, k2: float, device: torch.device, dtype=torch.float64) -> torch.Tensor:
    """K_{sr} for A → B → C, shape (S, S): row s says what feeds species s and what drains it."""
    return torch.tensor([[-k1, 0.0, 0.0], [k1, -k2, 0.0], [0.0, k2, 0.0]], device=device, dtype=dtype)


def concentrations(K: torch.Tensor, c0: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """c_{ts} = [exp(K t_t)]_{sr} c_r(0).   K: (S, S), c0: (S,), t: (T,) → (T, S).  r is summed."""
    propagators = torch.linalg.matrix_exp(K[None] * t[:, None, None])  # (T, S, S), exact for a linear ODE
    return torch.einsum("tsr,r->ts", propagators, c0)


def analytic(k1: float, k2: float, a0: float, t: torch.Tensor) -> torch.Tensor:
    """Textbook solution for A → B → C (k1 ≠ k2), shape (T, 3), for comparison:

    A = a0 e^{-k1 t},   B = a0 k1/(k2 - k1) (e^{-k1 t} - e^{-k2 t}),   C = a0 - A - B
    """
    A = a0 * torch.exp(-k1 * t)
    B = a0 * k1 / (k2 - k1) * (torch.exp(-k1 * t) - torch.exp(-k2 * t))
    return torch.stack([A, B, a0 - A - B], dim=1)


def pseudo_inverse(E: torch.Tensor) -> torch.Tensor:
    """E⁺_{sw} = (G⁻¹)_{sr} E_{wr} with G_{rs} = E_{wr} E_{ws}.   E: (W, S) → (S, W).  Needs independent spectra."""
    G = torch.einsum("wr,ws->rs", E, E)  # (S, S) Gram matrix: overlaps between the species' spectra
    return torch.einsum("sr,wr->sw", torch.linalg.inv(G), E)


def recover(D: torch.Tensor, E_pinv: torch.Tensor, path_length: float) -> torch.Tensor:
    """ĉ_{ts} = E⁺_{sw} D_{tw} / l: least-squares concentrations.   D: (T, W) → (T, S).  w is summed."""
    return torch.einsum("sw,tw->ts", E_pinv, D) / path_length


def crossings(lam: torch.Tensor, eps_a: torch.Tensor, eps_b: torch.Tensor, floor: float = 0.02) -> list[float]:
    """Wavelengths where two spectra cross (ε_a = ε_b): candidate isosbestic points, by linear interpolation.

    Crossings where both absorb less than `floor` × the larger peak are ignored (two tails, no signal).
    """
    diff = eps_a - eps_b
    level = floor * torch.maximum(eps_a.max(), eps_b.max())
    found = []
    for w in torch.nonzero(diff[:-1] * diff[1:] < 0).flatten().tolist():
        frac = (diff[w] / (diff[w] - diff[w + 1])).item()
        if max(eps_a[w].item(), eps_b[w].item()) > level:
            found.append(lam[w].item() + frac * (lam[w + 1] - lam[w]).item())
    return found


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/beer_lambert_kinetics/tensors.py
"""Build the concrete tensors: wavelength grid, molar absorptivities ε_ws, rate matrix, sampling times.

The spectra are synthetic: each species gets a Gaussian absorption band with a textbook-sized molar absorptivity
(10⁴ L mol⁻¹ cm⁻¹ is typical for a strongly coloured organic dye or a metal complex).
"""

import torch

DTYPE = torch.float64
SPECIES = ("A", "B", "C")
PATH_LENGTH = 1.0  # cuvette path length l (cm)
N_SPECTRA = 121  # spectra recorded during the run

# synthetic bands: peak wavelength (nm), width σ (nm), peak ε (L mol⁻¹ cm⁻¹)
BANDS = {
    "A": [(350.0, 30.0, 12000.0)],
    "B": [(450.0, 40.0, 9000.0)],
    "C": [(560.0, 45.0, 15000.0)],
}


def wavelengths(device: torch.device) -> torch.Tensor:
    """λ_w in nm, shape (W,): 300–700 nm every 2 nm."""
    return torch.linspace(300.0, 700.0, 201, device=device, dtype=DTYPE)


def absorptivities(lam: torch.Tensor) -> torch.Tensor:
    """E_ws = ε_s(λ_w) in L mol⁻¹ cm⁻¹, shape (W, S): one column per species, a sum of Gaussian bands."""
    columns = []
    for name in SPECIES:
        eps = torch.zeros_like(lam)
        for peak, width, height in BANDS[name]:
            eps = eps + height * torch.exp(-0.5 * ((lam - peak) / width) ** 2)
        columns.append(eps)
    return torch.stack(columns, dim=1)


def initial_concentrations(c0_uM: float, device: torch.device) -> torch.Tensor:
    """c_s(0) in mol/L, shape (S,): only A at the start."""
    return torch.tensor([c0_uM * 1e-6, 0.0, 0.0], device=device, dtype=DTYPE)


def sample_times(t_end: float, device: torch.device) -> torch.Tensor:
    """t_t in s, shape (T,): when the spectrophotometer takes a spectrum."""
    return torch.linspace(0.0, t_end, N_SPECTRA, device=device, dtype=DTYPE)


def noise(shape: tuple[int, ...], sigma: float, device: torch.device, seed: int = 0) -> torch.Tensor:
    """Gaussian detector noise in absorbance units, with a fixed seed (the same numbers on every device)."""
    gen = torch.Generator().manual_seed(seed)
    return sigma * torch.randn(shape, generator=gen, dtype=DTYPE).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/beer_lambert_kinetics/solve.py
"""Simulate the experiment (kinetics → spectra + noise) and analyse it (spectra → concentrations)."""

import torch

from . import equation


def simulate(
    E: torch.Tensor, c0: torch.Tensor, t: torch.Tensor, k1: float, k2: float, path_length: float, noise: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the true concentrations c (T, S) and the measured spectra D (T, W) = l E c + noise."""
    K = equation.rate_matrix(k1, k2, c0.device, c0.dtype)
    c = equation.concentrations(K, c0, t)
    return c, equation.absorbance(E, c, path_length) + noise


def unmix(E: torch.Tensor, D: torch.Tensor, path_length: float) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Least-squares concentrations from the spectra.   Returns (ĉ (T, S), residual spectra (T, W), cond(E))."""
    c_hat = equation.recover(D, equation.pseudo_inverse(E), path_length)
    residual = D - equation.absorbance(E, c_hat, path_length)
    return c_hat, residual, torch.linalg.cond(E).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/beer_lambert_kinetics/plot.py
"""Animate the run: the pure spectra, the measured spectra changing in time, and the recovered concentrations."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

COLORS = ["#2563eb", "#16a34a", "#dc2626"]  # A, B, C
N_HISTORY = 12  # faint earlier spectra kept on screen


def animate(
    lam: torch.Tensor,
    E: torch.Tensor,
    t: torch.Tensor,
    c: torch.Tensor,
    c_hat: torch.Tensor,
    D: torch.Tensor,
    iso_ab: list[float],
    iso_bc: list[float],
    path: Path,
) -> Path:
    lam, E, t = lam.cpu().numpy(), E.cpu().numpy(), t.cpu().numpy()
    c, c_hat, D = c.cpu().numpy() * 1e6, c_hat.cpu().numpy() * 1e6, D.cpu().numpy()  # µM
    n_t = len(t)

    fig = plt.figure(figsize=(8.6, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.1, 1], height_ratios=[0.8, 1.2])
    ax_e = fig.add_subplot(grid[0, 0])
    ax_d = fig.add_subplot(grid[1, 0])
    ax_c = fig.add_subplot(grid[:, 1])

    # --- pure-component spectra (columns of E) and their crossings ---
    for s, name in enumerate("ABC"):
        ax_e.plot(lam, E[:, s] / 1e3, color=COLORS[s], lw=1.6, label=name)
    for ax in (ax_e, ax_d):
        for x in iso_ab:
            ax.axvline(x, color="#f59e0b", ls="--", lw=1)
        for x in iso_bc:
            ax.axvline(x, color="#9333ea", ls=":", lw=1.2)
    ax_e.set_xlim(lam[0], lam[-1])
    ax_e.set_ylim(0, E.max() / 1e3 * 1.35)  # headroom for the legend
    ax_e.set_ylabel(r"$\varepsilon$ ($10^3$ L/(mol cm))", fontsize=8)
    title = r"columns of $E$: pure spectra   (-- $\varepsilon_A=\varepsilon_B$,  ⋯ $\varepsilon_B=\varepsilon_C$)"
    ax_e.set_title(title, fontsize=8.5, pad=2)
    ax_e.legend(fontsize=7, ncol=3, loc="upper right")
    ax_e.tick_params(labelsize=7)
    ax_e.set_xticklabels([])

    # --- measured spectra D(λ, t): faint history + the current one ---
    ax_d.set_xlim(lam[0], lam[-1])
    ax_d.set_ylim(min(0.0, D.min()) - 0.02, D.max() * 1.1)
    ax_d.set_xlabel("wavelength λ (nm)")
    ax_d.set_ylabel("absorbance D")
    ax_d.tick_params(labelsize=8)
    history_steps = np.linspace(0, n_t - 1, N_HISTORY).round().astype(int)
    shade = plt.get_cmap("viridis")(np.linspace(0, 0.9, N_HISTORY))
    history = [ax_d.plot([], [], lw=0.8, color=shade[i], alpha=0.6)[0] for i in range(N_HISTORY)]
    (current,) = ax_d.plot([], [], lw=1.8, color="#111827")
    clock = ax_d.text(0.98, 0.95, "", transform=ax_d.transAxes, ha="right", va="top", fontsize=9)

    # --- concentrations: true (lines) vs recovered from the spectra (dots) ---
    ax_c.set_xlim(0, t[-1])
    ax_c.set_ylim(min(0.0, c_hat.min()) - 1, c[0].sum() * 1.12)
    ax_c.set_xlabel("time t (s)")
    ax_c.set_ylabel("concentration (µM)")
    ax_c.set_title(r"recovered $\hat c = E^{+} D / l$ (dots) vs true (lines)", fontsize=9)
    for s, name in enumerate("ABC"):
        ax_c.plot(t, c[:, s], color=COLORS[s], lw=1.4, alpha=0.8, label=f"[{name}]")
    ax_c.plot(t, c.sum(1), color="#374151", ls="--", lw=1, label="[A]+[B]+[C]")
    dots = [ax_c.plot([], [], "o", ms=2.5, color=COLORS[s])[0] for s in range(3)]
    (dots_sum,) = ax_c.plot([], [], ".", ms=2.5, color="#374151")
    cursor = ax_c.axvline(0, color="#9ca3af", lw=1)
    ax_c.legend(fontsize=7, loc="center right")

    def update(step: int):
        for i, h in enumerate(history_steps):
            if h <= step:
                history[i].set_data(lam, D[h])
            else:
                history[i].set_data([], [])
        current.set_data(lam, D[step])
        clock.set_text(f"t = {t[step]:.0f} s")
        for s in range(3):
            dots[s].set_data(t[: step + 1], c_hat[: step + 1, s])
        dots_sum.set_data(t[: step + 1], c_hat[: step + 1].sum(1))
        cursor.set_xdata([t[step], t[step]])

    fig.tight_layout(pad=0.6, h_pad=0.4, w_pad=1.0)
    return save_gif(fig, update, frame_indices(n_t, 61), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'k1': 0.02,
    'k2': 0.003,
    't_end': 1200.0,
    'c0': 50.0,
    'noise': 0.003
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.beer_lambert_kinetics")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.beer_lambert_kinetics.main")
problem_dir = Path("problems/physchem/beer_lambert_kinetics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
